In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from toponymy.annotation import NodeId, AnnotationTree, Annotation, AnnotationStore, Executor

12:44:32 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
12:44:32 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [3]:
import numpy as np

In [4]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

In [5]:
from toponymy import ToponymyClusterer
clusterer = ToponymyClusterer(min_clusters=4, verbose=True)
clusterer.fit(document_map, embeddings);

Layer 0 found 8 clusters
Layer 1 found 4 clusters


In [6]:
clusterer.cluster_tree_

{(1, 0): [(0, 0)],
 (1, 1): [(0, 1)],
 (1, 2): [(0, 7), (0, 6), (0, 5)],
 (1, 3): [(0, 4)],
 (2, 0): [(1, 0), (1, 1), (1, 2), (1, 3), (0, 2), (0, 3)]}

## Nodes

In [7]:
node = NodeId(1, 0)

In [8]:
node.layer

1

In [9]:
node.cluster

0

In [10]:
node == (1, 0)

True

## AnnotationTree

Here are the assumptions about the condensed tree that must hold.

* There is exactly one root node.
* Every non-root node has exactly one parent.
* Edges may span more than one layer.
* The root is a catch-all node, and will not be considered a cluster node or a part of a layer.
* Layer ids are 0-indexed and there are no empty layers (this assumption is used by the executor)



Can make it from a condensed tree dict

In [11]:
tree = AnnotationTree(clusterer.cluster_tree_)

Or for an object with a `.cluster_tree_`

In [12]:
tree_from_clusterer = AnnotationTree.from_clusterer(clusterer)

Can check equality

In [13]:
tree == tree_from_clusterer

True

Can check if a node is in the tree

In [14]:
(0, 1) in tree

True

The root is not considered to be in the annotation tree

In [15]:
(2, 0) in tree

False

In [16]:
len(tree)

12

### Nodes
Has an iterable of all of the cluster nodes (the non-root nodes)

In [17]:
tree.nodes

(NodeId(0, 0),
 NodeId(0, 1),
 NodeId(0, 2),
 NodeId(0, 3),
 NodeId(0, 4),
 NodeId(0, 5),
 NodeId(0, 6),
 NodeId(0, 7),
 NodeId(1, 0),
 NodeId(1, 1),
 NodeId(1, 2),
 NodeId(1, 3))

In [18]:
len(tree) == len(tree.nodes)

True

### Layers
Gives access to the layers via their ids

In [19]:
tree.n_layers

2

In [20]:
tree.layer_ids

(0, 1)

In [21]:
tree.layer(0)

(NodeId(0, 0),
 NodeId(0, 1),
 NodeId(0, 2),
 NodeId(0, 3),
 NodeId(0, 4),
 NodeId(0, 5),
 NodeId(0, 6),
 NodeId(0, 7))

### Relatives

In [22]:
root_node = (2, 0)
leaf_node = (0, 0)
top_layer_node = (1, 2)

In [23]:
tree.parent(leaf_node)

NodeId(1, 0)

In [24]:
tree.parent(top_layer_node)

In [25]:
tree.children(leaf_node)

In [26]:
tree.children(top_layer_node)

[NodeId(0, 5), NodeId(0, 6), NodeId(0, 7)]

In [27]:
tree.descendants(top_layer_node)

[NodeId(0, 5), NodeId(0, 6), NodeId(0, 7)]

In [28]:
tree.descendants(leaf_node)

[]

The root is hidden from parent/child relationships since it doesn't represent a cluster

In [29]:
tree.parent(root_node)

In [30]:
tree.children(root_node)

In [31]:
tree.descendants(root_node)

[]

If you need to see the root or root children for some reason, there's internal access to them

In [32]:
tree._root

NodeId(2, 0)

In [33]:
tree._root_children

[NodeId(0, 2),
 NodeId(0, 3),
 NodeId(1, 0),
 NodeId(1, 1),
 NodeId(1, 2),
 NodeId(1, 3)]

## Annotation
An `Annotation` is a data store for annotations of the nodes of an `AnnotationTree`

It holds up to one object for each node in an `AnnotationTree`

In [34]:
example = Annotation("example", tree)

Here's an empty Annotation

In [35]:
example.states

{NodeId(0, 0): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 1): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 2): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 3): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 4): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 5): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 6): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 7): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 0): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 1): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 2): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 3): <AnnotationState.EMPTY: 'empty'>}

or the friendlier version:

In [36]:
example.get(leaf_node)

You can fill in an annotation from a layered list (aka. list of lists indexed by layer id and then cluster id)

In [37]:
# note, no need to cooerce the array to lists, except to be able to show equality below
centroid_vectors_layered_list = [list(clusterer.cluster_layers_[0].centroid_vectors), list(clusterer.cluster_layers_[1].centroid_vectors)]

In [38]:
centroid_annotation = Annotation.from_layered_list("centroids", tree, centroid_vectors_layered_list)

In [39]:
centroid_annotation.states

{NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

In [40]:
node = NodeId(1, 0)

In [41]:
all(clusterer.cluster_layers_[node.layer].centroid_vectors[node.cluster] == centroid_annotation[node])

True

And get a layered list of lists back from the annotation as long as all of the nodes have been computed. This gives backwards compatibility with the old representations. 

In [42]:
new_layered_list = centroid_annotation.to_layered_list()

In [43]:
centroid_vectors_layered_list == new_layered_list

True

## AnnotationStore

In [44]:
store = AnnotationStore(tree, [centroid_annotation])

In [45]:
store

AnnotationStore(centroids[12/12])

In [46]:
all(store['centroids'][node] == store.centroids[node])

True

In [47]:
all(store.centroids[node] == store.node(node)['centroids'])

True

In [48]:
store.node_states(node)

{'centroids': <AnnotationState.COMPUTED: 'computed'>}

## Annotator

In [49]:
class FirstEntryAnnotator:
    inputs = ("centroids",)
    outputs = ("first_item",)
    algorithm_type = "node-node"

    def annotate(
        self,
        node,
        *,
        centroids # has to match inputs name
    ):
        return {"first_item" : centroids[0]} # keys have to match outputs

## Executor

In [50]:
executor = Executor(store)

In [51]:
failures = executor.run(FirstEntryAnnotator())

In [52]:
failures

{}

In [53]:
store

AnnotationStore(centroids[12/12], first_item[12/12])

In [54]:
store['first_item'][node]

np.float64(-0.010361493309028446)

In [55]:
store['centroids'][node][0]

np.float64(-0.010361493309028446)

## Exemplar

Already have the centroid annotation so now we need: 

* cluster objects annotation
* cluster object vectors annotation

In [56]:
from toponymy.exemplar_texts import diverse_exemplars_by_cluster

In [57]:
cluster_label_vectors_list = [clusterer.cluster_layers_[0].cluster_labels, clusterer.cluster_layers_[1].cluster_labels]

### Create `cluster_object_annotation` from `document_map`

In [58]:
object_list = document_map

In [59]:
#def from_cluster_list(objects, cluster_label_vectors_list):
cluster_object_annotation = Annotation("cluster_objects", tree)
for layer_id, cluster_label_vector in enumerate(cluster_label_vectors_list):
    for cluster_id in range(cluster_label_vector.max() + 1):
        cluster_mask = cluster_label_vector == cluster_id
        original_indices = np.where(cluster_mask)[0]
        cluster_object_annotation[layer_id, cluster_id] = [object_list[i] for i in original_indices]
cluster_object_annotation
    

In [60]:
#cluster_object_annotation = from_cluster_list(objects, cluster_label_vectors_list)

In [61]:
cluster_object_annotation.states

{NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

In [62]:
cluster_object_annotation[0, 0]

[array([-0.031685  ,  8.11856461]),
 array([-1.48840129,  8.61860752]),
 array([-1.43386745,  7.64857864]),
 array([-0.56737387,  9.60540009]),
 array([-0.61449176,  8.44878674]),
 array([-0.81579286,  7.50361252]),
 array([-0.76751059,  7.53403664]),
 array([-0.80946195,  8.22869587]),
 array([-1.45300865,  7.68642044]),
 array([-0.79968768,  7.49941111]),
 array([-5.90173760e-03,  8.51672935e+00]),
 array([-0.09859695,  9.77624989]),
 array([-0.47076595,  9.5443325 ]),
 array([-0.46936759,  9.67712116]),
 array([-0.32179111,  9.4757576 ]),
 array([-1.0466094 ,  7.85938263]),
 array([-0.23639934,  9.46710014]),
 array([-0.55644304,  7.78719282]),
 array([-0.30645669, 10.36446667]),
 array([-1.01562333,  7.92516661])]

In [63]:
store.add(cluster_object_annotation)

### Create `cluster_object_vectors_annotation` from `object_vectors`

In [64]:
object_vectors = embeddings

In [65]:
cluster_object_vectors_annotation = Annotation("cluster_object_vectors", tree)
null_topic = np.mean(object_vectors, axis=0)

for layer_id, cluster_label_vector in enumerate(cluster_label_vectors_list):
    for cluster_id in range(cluster_label_vector.max() + 1):
        cluster_mask = cluster_label_vector == cluster_id
        cluster_object_vectors_annotation[layer_id, cluster_id] = object_vectors[cluster_mask] - null_topic

In [66]:
cluster_object_vectors_annotation.states

{NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

In [67]:
cluster_object_vectors_annotation[0,0]

array([[-0.03516523, -0.02470507, -0.00457667, ..., -0.00025323,
         0.00974339, -0.01337929],
       [-0.02701342,  0.02090386, -0.0088631 , ..., -0.02663631,
        -0.04598448, -0.02278181],
       [-0.04193112,  0.03203582, -0.00346031, ...,  0.0042168 ,
         0.05402277, -0.01278228],
       ...,
       [-0.01031948, -0.05649849, -0.03335569, ...,  0.03093364,
         0.06822366,  0.0142484 ],
       [-0.03946716, -0.04731105, -0.03005292, ..., -0.03029699,
         0.00874201, -0.01366989],
       [-0.02024932,  0.0785161 ,  0.00673887, ...,  0.08301054,
         0.02758984, -0.01715851]], shape=(20, 768))

In [68]:
store.add(cluster_object_vectors_annotation, replace=True)

## Compute exemplars

raw version

In [69]:
store

AnnotationStore(centroids[12/12], first_item[12/12], cluster_objects[12/12], cluster_object_vectors[12/12])

In [70]:
np.array(store["cluster_objects"][node])

array([[-3.16849984e-02,  8.11856461e+00],
       [-1.48840129e+00,  8.61860752e+00],
       [-1.43386745e+00,  7.64857864e+00],
       [-5.67373872e-01,  9.60540009e+00],
       [-6.14491761e-01,  8.44878674e+00],
       [-8.15792859e-01,  7.50361252e+00],
       [-7.67510593e-01,  7.53403664e+00],
       [-8.09461951e-01,  8.22869587e+00],
       [-1.45300865e+00,  7.68642044e+00],
       [-7.99687684e-01,  7.49941111e+00],
       [-5.90173760e-03,  8.51672935e+00],
       [-9.85969454e-02,  9.77624989e+00],
       [-4.70765948e-01,  9.54433250e+00],
       [-4.69367594e-01,  9.67712116e+00],
       [-3.21791112e-01,  9.47575760e+00],
       [-1.04660940e+00,  7.85938263e+00],
       [-2.36399338e-01,  9.46710014e+00],
       [-5.56443036e-01,  7.78719282e+00],
       [-3.06456685e-01,  1.03644667e+01],
       [-1.01562333e+00,  7.92516661e+00]])

In [71]:
results = []
indices = []

for node in tree.nodes:
    layer = clusterer.cluster_layers_[node.layer]
    node_result = diverse_exemplars_by_cluster(cluster_objects=np.array(store["cluster_objects"][node]),
                                               cluster_centroid=store["centroids"][node],
                                               cluster_object_vectors=store["cluster_object_vectors"][node], 
                                               null_topic=null_topic,
                                               n_exemplars=layer.n_exemplars,
                                               diversify_alpha=layer.exemplars_diversify_alpha,
                                               object_to_text_function=layer.object_to_text_function,
                                               verbose=layer.verbose,
                                               show_progress_bar=layer.show_progress_bar)
    chosen_exemplars, exemplar_order, chosen_indices  = node_result
    cluster_mask = layer.cluster_labels == node.cluster
    original_indices = np.where(cluster_mask)[0]
    chosen_original_indices = [
        original_indices[exemplar_order[i]] for i in chosen_indices
    ]
    
    results.append(chosen_exemplars)
    indices.append(chosen_original_indices)


In [72]:
layer.exemplars_diversify_alpha

1.0

In [73]:
class DiverseExemplarAnnotator:
    inputs = ("cluster_objects", "centroids", "cluster_object_vectors",)
    outputs = ("exemplars", "examplar_original_indices")
    algorithm_type = "node-node"

    def __init__(self, null_topic=None, n_exemplars=None, diversify_alpha=None, object_to_text_function=None, cluster_label_vectors_list=None):
        self.null_topic=null_topic
        self.n_exemplars=n_exemplars
        self.diversify_alpha=diversify_alpha
        self.object_to_text_function=object_to_text_function
        self.cluster_label_vectors_list=cluster_label_vectors_list
        print(self.n_exemplars)

    def annotate(
        self,
        node,
        *,
        cluster_objects,
        centroids,
        cluster_object_vectors,# has to match inputs name
    ):

        node_result = diverse_exemplars_by_cluster(cluster_objects=cluster_objects,
                                                   cluster_centroid=centroids,
                                                   cluster_object_vectors=cluster_object_vectors, 
                                                   null_topic=self.null_topic,
                                                   n_exemplars=self.n_exemplars,
                                                   diversify_alpha=self.diversify_alpha,
                                                   object_to_text_function=self.object_to_text_function)
        chosen_exemplars, exemplar_order, chosen_indices  = node_result
        cluster_mask = cluster_label_vectors_list[node.layer] == node.cluster
        original_indices = np.where(cluster_mask)[0]
        chosen_original_indices = [
            original_indices[exemplar_order[i]] for i in chosen_indices
        ]
        
        return {"exemplars" : chosen_exemplars, "examplar_original_indices": original_indices} # keys have to match outputs

In [74]:
#%debug
executor.run(DiverseExemplarAnnotator(null_topic=null_topic,
                                      n_exemplars=8,
                                      diversify_alpha=layer.exemplars_diversify_alpha, 
                                      object_to_text_function=layer.object_to_text_function, 
                                      cluster_label_vectors_list=cluster_label_vectors_list))

8


{}

In [75]:
executor.store.exemplars[0,0]

[array([-1.45300865,  7.68642044]),
 array([-0.80946195,  8.22869587]),
 array([-0.46936759,  9.67712116]),
 array([-0.81579286,  7.50361252]),
 array([-0.30645669, 10.36446667]),
 array([-0.32179111,  9.4757576 ]),
 array([-1.0466094 ,  7.85938263]),
 array([-1.48840129,  8.61860752])]

In [76]:
results[0]

[array([-1.45300865,  7.68642044]),
 array([-0.80946195,  8.22869587]),
 array([-0.46936759,  9.67712116]),
 array([-0.81579286,  7.50361252]),
 array([-0.30645669, 10.36446667]),
 array([-0.32179111,  9.4757576 ]),
 array([-1.0466094 ,  7.85938263]),
 array([-1.48840129,  8.61860752])]

In [77]:
indices[0]

[np.int64(74),
 np.int64(66),
 np.int64(101),
 np.int64(54),
 np.int64(134),
 np.int64(111),
 np.int64(115),
 np.int64(3)]